# 07 — Paper Results: Chapter 4 (Results and Discussion)

This notebook generates **all quantitative values, tables, and figures** for Chapter 4 of the journal paper on hybrid bias correction of satellite precipitation (IMERG) for Indonesia.

### Outputs generated

| Output | Section | Description |
|--------|---------|-------------|
| **Table 4** | 4.1 | Progressive improvement across correction stages (vs CPC-UNI) |
| **Stratified analysis** | 4.1 | Station-covered vs station-sparse performance split |
| **Table 5** | 4.3 | Precipitation percentile distribution |
| **Table 6** | 4.4 | Full 9-combination reference-test evaluation matrix |
| **Figure 4** | 4.2 | CQI spatial distribution maps (3-panel) |
| **Figure 5** | 4.2 | CQI improvement map + categorical classification |
| **Figure 6** | 4.3 | CQI component score box plots |
| **Table 7** | 4.5 | Station-level validation summary |
| **Figure 7** | 4.5 | Station locations + per-station NSE map |
| **Figure 8** | 4.5 | WMO multi-threshold verification curves |
| **Figure 9** | 4.6 | Confidence mask analysis |
| **Seasonal summary** | Supp. | Wet vs dry season CQI |
| **Temporal stability** | Supp. | Per-year RMSE timeseries |
| **Narrative values** | All | Fill-in numbers for paper text |

### Prerequisites

Notebooks 02 (bias correction), 03 (metrics), 04 (QA framework), and 06 (station validation) must be completed for all 36 dekadal periods before running this notebook.

### Consistent metric set across tables

Eight metrics are reported in Tables 4, 6, and 7: RB, Pearson Corr, RMSE, MAE, NSE, POD, CSI, KS p-value.

In [ ]:
# ==============================================================
# Section 0: Setup
# ==============================================================

import os
import sys
import platform
import warnings
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Windows DLL fix for conda environments
if platform.system() == 'Windows':
    try:
        import ctypes
        ctypes.cdll.LoadLibrary('hdf5.dll')
    except Exception:
        pass
    _conda_prefix = os.environ.get('CONDA_PREFIX') or sys.prefix
    _dll_dirs = [
        os.path.join(_conda_prefix, 'Library', 'bin'),
        os.path.join(_conda_prefix, 'Library', 'lib'),
    ]
    for _d in _dll_dirs:
        if os.path.isdir(_d):
            try:
                os.add_dll_directory(_d)
            except OSError:
                pass
            if _d not in os.environ.get('PATH', ''):
                os.environ['PATH'] = _d + os.pathsep + os.environ.get('PATH', '')

# Project root detection
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.isfile(os.path.join(project_root, 'src', 'config.py')):
    # Fallback: try current directory
    project_root = os.getcwd()
assert os.path.isfile(os.path.join(project_root, 'src', 'config.py')), \
    f"Project root not found at {project_root}"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Reload src modules cleanly
for m in [k for k in sys.modules if k.startswith('src')]:
    del sys.modules[m]

import importlib
import src.config as _cfg
importlib.reload(_cfg)
_cfg.initialize_config(os.path.join(project_root, 'config.yml'))

from src import config

# ---------------------------------------------------------------
# Paper-level constants
# ---------------------------------------------------------------

# 8 paper metrics (consistent across Tables 4, 6, 7)
PAPER_METRICS = [
    'relative_bias', 'pearson_correlation', 'rmse', 'mae',
    'nse', 'pod', 'csi', 'ks_pvalue',
]
PAPER_METRIC_LABELS = ['RB', 'Corr', 'RMSE', 'MAE', 'NSE', 'POD', 'CSI', 'KS p']

METHODS = ['ls', 'lseqm', 'lseqmdl']
METHOD_LABELS = ['LS', 'LSEQM', 'LSEQM+DL']
REFS = ['cpc', 'imergl', 'imergf']

DEKAD_MAP = {1: '01', 2: '11', 3: '21'}
ALL_PERIODS = [(m, d) for m in range(1, 13) for d in [1, 2, 3]]
WET_MONTHS = [10, 11, 12, 1, 2, 3]
DRY_MONTHS = [4, 5, 6, 7, 8, 9]

# ---------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------

def get_metrics_dir(method):
    return config.metrics_path_template.replace('{method}', method)


def get_quality_dir(method):
    return config.quality_path_template.replace('{method}', method)


def metricssd_path(method, ref, month, dekad):
    dd = DEKAD_MAP[dekad]
    mm = f"{month:02d}"
    fname = f"idn_cli_metricssd_{ref}_imergl_{method}_month{mm}_dekad{dd}.nc4"
    return os.path.join(get_metrics_dir(method), fname)


def qualitysd_path(method, ref, month, dekad):
    dd = DEKAD_MAP[dekad]
    mm = f"{month:02d}"
    fname = f"idn_cli_qualitysd_{ref}_imergl_{method}_month{mm}_dekad{dd}.nc4"
    return os.path.join(get_quality_dir(method), fname)


def station_val_path(method, month, dekad):
    dd = DEKAD_MAP[dekad]
    mm = f"{month:02d}"
    return os.path.join(
        config.STATION_VALIDATION_OUTPUT_DIR,
        f"station_validation_{method}_month{mm}_dekad{dd}.csv",
    )


def multi_thresh_path(method, month, dekad):
    dd = DEKAD_MAP[dekad]
    mm = f"{month:02d}"
    return os.path.join(
        config.STATION_VALIDATION_OUTPUT_DIR,
        f"multi_threshold_summary_{method}_month{mm}_dekad{dd}.csv",
    )


def spatial_stats(ds, var_names):
    """Compute spatial median, Q25, Q75 across (lat, lon) for each variable."""
    result = {}
    for v in var_names:
        if v not in ds:
            result[v] = {'median': np.nan, 'q25': np.nan, 'q75': np.nan}
            continue
        vals = ds[v].values.ravel()
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            result[v] = {'median': np.nan, 'q25': np.nan, 'q75': np.nan}
        else:
            result[v] = {
                'median': float(np.median(vals)),
                'q25': float(np.percentile(vals, 25)),
                'q75': float(np.percentile(vals, 75)),
            }
    return result


def load_and_summarize_all_dekads(method, ref, var_names):
    """Load metricssd for all 36 periods, return list of spatial_stats dicts."""
    summaries = []
    for month, dekad in ALL_PERIODS:
        fpath = metricssd_path(method, ref, month, dekad)
        if not os.path.isfile(fpath):
            continue
        ds = xr.open_dataset(fpath, engine=config.NETCDF_ENGINE)
        s = spatial_stats(ds, var_names)
        s['_month'] = month
        s['_dekad'] = dekad
        summaries.append(s)
        ds.close()
    return summaries


def aggregate_summaries(summaries, var_names):
    """Median of spatial medians across dekads."""
    result = {}
    for v in var_names:
        medians = [s[v]['median'] for s in summaries if np.isfinite(s[v]['median'])]
        q25s = [s[v]['q25'] for s in summaries if np.isfinite(s[v]['q25'])]
        q75s = [s[v]['q75'] for s in summaries if np.isfinite(s[v]['q75'])]
        result[v] = {
            'median': float(np.median(medians)) if medians else np.nan,
            'q25': float(np.median(q25s)) if q25s else np.nan,
            'q75': float(np.median(q75s)) if q75s else np.nan,
        }
    return result


# Figure output directory
figures_dir = os.path.join(config.output_dir, 'figures', 'paper')
os.makedirs(figures_dir, exist_ok=True)


def save_fig(fig, name):
    fpath = os.path.join(figures_dir, name)
    fig.savefig(fpath, dpi=300, bbox_inches='tight')
    print(f"Saved: {fpath}")


print(f"Project root:   {project_root}")
print(f"Output dir:     {config.output_dir}")
print(f"Figures dir:    {figures_dir}")
print(f"NetCDF engine:  {config.NETCDF_ENGINE}")
print(f"Metrics path:   {config.metrics_path_template}")
print(f"Quality path:   {config.quality_path_template}")
print(f"Station val:    {config.STATION_VALIDATION_OUTPUT_DIR}")

### Step 1: Verify Pre-Computed Outputs

Check that notebooks 03, 04, and 06 have been completed for all 36 dekadal periods.

In [ ]:
# ==============================================================
# Step 1: Data inventory
# ==============================================================

inventory = {'metricssd': {}, 'qualitysd': {}, 'station_val': {}}

for method in METHODS:
    count_m = 0
    count_q = 0
    count_s = 0
    for month, dekad in ALL_PERIODS:
        # Metrics (CPC reference)
        if os.path.isfile(metricssd_path(method, 'cpc', month, dekad)):
            count_m += 1
        # Quality (CPC reference)
        if os.path.isfile(qualitysd_path(method, 'cpc', month, dekad)):
            count_q += 1
        # Station validation
        if os.path.isfile(station_val_path(method, month, dekad)):
            count_s += 1
    inventory['metricssd'][method] = count_m
    inventory['qualitysd'][method] = count_q
    inventory['station_val'][method] = count_s

# Also check other references for metrics
for ref in ['imergl', 'imergf']:
    for method in METHODS:
        count = sum(
            1 for month, dekad in ALL_PERIODS
            if os.path.isfile(metricssd_path(method, ref, month, dekad))
        )
        inventory['metricssd'][f'{method}_{ref}'] = count

print("DATA INVENTORY (out of 36 dekadal periods)")
print("=" * 65)
print(f"{'Dataset':>25s} | {'LS':>5s} | {'LSEQM':>5s} | {'LSEQMDL':>7s}")
print("-" * 65)
print(f"{'metricssd (vs CPC)':>25s} | {inventory['metricssd']['ls']:>5d} | "
      f"{inventory['metricssd']['lseqm']:>5d} | {inventory['metricssd']['lseqmdl']:>7d}")
print(f"{'metricssd (vs IMERGL)':>25s} | {inventory['metricssd'].get('ls_imergl', 0):>5d} | "
      f"{inventory['metricssd'].get('lseqm_imergl', 0):>5d} | {inventory['metricssd'].get('lseqmdl_imergl', 0):>7d}")
print(f"{'metricssd (vs IMERGF)':>25s} | {inventory['metricssd'].get('ls_imergf', 0):>5d} | "
      f"{inventory['metricssd'].get('lseqm_imergf', 0):>5d} | {inventory['metricssd'].get('lseqmdl_imergf', 0):>7d}")
print(f"{'qualitysd (vs CPC)':>25s} | {inventory['qualitysd']['ls']:>5d} | "
      f"{inventory['qualitysd']['lseqm']:>5d} | {inventory['qualitysd']['lseqmdl']:>7d}")
print(f"{'station_validation':>25s} | {inventory['station_val']['ls']:>5d} | "
      f"{inventory['station_val']['lseqm']:>5d} | {inventory['station_val']['lseqmdl']:>7d}")

# Warn if incomplete
for key in ['metricssd', 'qualitysd', 'station_val']:
    for method in METHODS:
        n = inventory[key][method]
        if n < 36:
            print(f"  WARNING: {key}/{method} has only {n}/36 periods")

### Step 2: Compute Uncorrected Baseline (CPC vs IMERG-L)

**Chapter 4.1** -- These metrics establish the magnitude of raw satellite bias before correction. Uses `compute_single_dekad_metrics()` from `src.metrics`.

In [ ]:
# ==============================================================
# Step 2: Uncorrected baseline (CPC vs IMERG-L)
# ==============================================================

from src.metrics import unify_cpc_for_metrics, slice_month_dekad, compute_single_dekad_metrics
from src.utility import apply_land_sea_mask
from src import config

# Load CPC + IMERG-L once
cpc_ds = xr.open_dataset(config.cpc_file, engine=config.NETCDF_ENGINE)
if cpc_ds.lat[0] > cpc_ds.lat[-1]:
    cpc_ds = cpc_ds.reindex(lat=cpc_ds.lat[::-1])

imergl_ds = xr.open_dataset(config.imergl_file, engine=config.NETCDF_ENGINE)
if imergl_ds.lat[0] > imergl_ds.lat[-1]:
    imergl_ds = imergl_ds.reindex(lat=imergl_ds.lat[::-1])

# Regrid CPC to IMERG grid
cpc_regridded = unify_cpc_for_metrics(cpc_ds, imergl_ds)

# Compute baseline for each dekad
baseline_summaries = []
for idx, (month, dekad) in enumerate(ALL_PERIODS, 1):
    ref_slice = slice_month_dekad(cpc_regridded[config.CPC_PRECIP_VAR], month, dekad)
    test_slice = slice_month_dekad(imergl_ds[config.IMERG_PRECIP_VAR], month, dekad)
    ref_slice = apply_land_sea_mask(ref_slice, config.mask_file)
    test_slice = apply_land_sea_mask(test_slice, config.mask_file)

    metrics_ds = compute_single_dekad_metrics(ref_slice, test_slice)
    s = spatial_stats(metrics_ds, PAPER_METRICS)
    s['_month'] = month
    s['_dekad'] = dekad
    baseline_summaries.append(s)
    if idx % 6 == 0:
        print(f"  Baseline: {idx}/36 dekads done")

baseline_agg = aggregate_summaries(baseline_summaries, PAPER_METRICS)

print("\n--- Uncorrected IMERG-L Baseline (CPC vs IMERG-L) ---")
for m in PAPER_METRICS:
    v = baseline_agg[m]
    print(f"  {m:25s}: {v['median']:.4f}  (IQR: {v['q25']:.4f} - {v['q75']:.4f})")

cpc_ds.close()
imergl_ds.close()

### Step 3: Table 4 -- Progressive Improvement Across Correction Stages

**Chapter 4.1** -- Domain-averaged metrics for IMERG-L (uncorrected), LS, LSEQM, LSEQM+DL against CPC-UNI.

In [ ]:
# ==============================================================
# Step 3: Table 4 -- Domain-averaged metrics by correction stage
# ==============================================================

table4 = {'IMERG-L': baseline_agg}
for method, label in zip(METHODS, METHOD_LABELS):
    summaries = load_and_summarize_all_dekads(method, 'cpc', PAPER_METRICS)
    table4[label] = aggregate_summaries(summaries, PAPER_METRICS)
    print(f"  {label}: loaded {len(summaries)} dekads")

# Format Table 4
print("\n" + "=" * 100)
print("TABLE 4: Domain-averaged verification metrics by correction stage (vs CPC-UNI)")
print("=" * 100)
header = (f"{'Metric':>10s} | {'IMERG-L':>20s} | {'LS':>20s} | "
          f"{'LSEQM':>20s} | {'LSEQM+DL':>20s} | {'Perfect':>7s}")
print(header)
print("-" * len(header))

perfect = {
    'relative_bias': 0, 'pearson_correlation': 1, 'rmse': 0, 'mae': 0,
    'nse': 1, 'pod': 1, 'csi': 1, 'ks_pvalue': 1,
}

for m, label in zip(PAPER_METRICS, PAPER_METRIC_LABELS):
    row = f"{label:>10s}"
    for key in ['IMERG-L'] + METHOD_LABELS:
        v = table4[key][m]
        row += f" | {v['median']:7.4f} ({v['q25']:.3f}-{v['q75']:.3f})"
    row += f" | {perfect[m]:>7d}"
    print(row)

# Narrative values
print("\n--- Narrative fill-in values ---")
for m, label in zip(PAPER_METRICS, PAPER_METRIC_LABELS):
    vals = [table4[k][m]['median'] for k in ['IMERG-L'] + METHOD_LABELS]
    print(f"  {label}: IMERG-L={vals[0]:.4f} -> LS={vals[1]:.4f} -> "
          f"LSEQM={vals[2]:.4f} -> LSEQM+DL={vals[3]:.4f}")

### Step 3b: Stratified Analysis by Station Density

**Chapter 4.1** -- Split by station-covered (C >= 0.5) vs station-sparse (C < 0.5) using confidence mask.

In [ ]:
# ==============================================================
# Step 3b: Stratified analysis using confidence mask
# ==============================================================

# Load confidence mask
conf_ds = xr.open_dataset(config.CONFIDENCE_MASK_FILE, engine=config.NETCDF_ENGINE)
conf_mask = conf_ds['confidence'].values  # 2D array
conf_lats = conf_ds['lat'].values
conf_lons = conf_ds['lon'].values
conf_ds.close()

station_covered_pct = np.sum(conf_mask >= 0.5) / np.sum(np.isfinite(conf_mask)) * 100
print(f"Station-covered pixels (C >= 0.5): {station_covered_pct:.1f}%")

# Build interpolated confidence DataArray once
conf_interp = xr.DataArray(
    conf_mask, dims=['lat', 'lon'],
    coords={'lat': conf_lats, 'lon': conf_lons},
)

# For each method, split metrics by confidence
stratified = {}
for method, label in zip(METHODS, METHOD_LABELS):
    covered_medians = {m: [] for m in PAPER_METRICS}
    sparse_medians = {m: [] for m in PAPER_METRICS}

    for month, dekad in ALL_PERIODS:
        fpath = metricssd_path(method, 'cpc', month, dekad)
        if not os.path.isfile(fpath):
            continue
        ds = xr.open_dataset(fpath, engine=config.NETCDF_ENGINE)
        conf_on_grid = conf_interp.interp(
            lat=ds.lat, lon=ds.lon, method='nearest'
        )

        for m in PAPER_METRICS:
            if m not in ds:
                continue
            vals = ds[m].values
            cmask = conf_on_grid.values
            covered_vals = vals[cmask >= 0.5]
            sparse_vals = vals[cmask < 0.5]
            covered_vals = covered_vals[np.isfinite(covered_vals)]
            sparse_vals = sparse_vals[np.isfinite(sparse_vals)]
            if len(covered_vals) > 0:
                covered_medians[m].append(np.median(covered_vals))
            if len(sparse_vals) > 0:
                sparse_medians[m].append(np.median(sparse_vals))
        ds.close()

    stratified[label] = {
        'covered': {
            m: np.median(covered_medians[m]) if covered_medians[m] else np.nan
            for m in PAPER_METRICS
        },
        'sparse': {
            m: np.median(sparse_medians[m]) if sparse_medians[m] else np.nan
            for m in PAPER_METRICS
        },
    }
    print(f"  {label}: done")

# Baseline stratification
baseline_covered = {m: [] for m in PAPER_METRICS}
baseline_sparse = {m: [] for m in PAPER_METRICS}

cpc_ds2 = xr.open_dataset(config.cpc_file, engine=config.NETCDF_ENGINE)
if cpc_ds2.lat[0] > cpc_ds2.lat[-1]:
    cpc_ds2 = cpc_ds2.reindex(lat=cpc_ds2.lat[::-1])
imergl_ds2 = xr.open_dataset(config.imergl_file, engine=config.NETCDF_ENGINE)
if imergl_ds2.lat[0] > imergl_ds2.lat[-1]:
    imergl_ds2 = imergl_ds2.reindex(lat=imergl_ds2.lat[::-1])
cpc_reg2 = unify_cpc_for_metrics(cpc_ds2, imergl_ds2)

for idx, (month, dekad) in enumerate(ALL_PERIODS, 1):
    ref_sl = slice_month_dekad(cpc_reg2[config.CPC_PRECIP_VAR], month, dekad)
    test_sl = slice_month_dekad(imergl_ds2[config.IMERG_PRECIP_VAR], month, dekad)
    ref_sl = apply_land_sea_mask(ref_sl, config.mask_file)
    test_sl = apply_land_sea_mask(test_sl, config.mask_file)
    mds = compute_single_dekad_metrics(ref_sl, test_sl)

    conf_on_grid2 = conf_interp.interp(
        lat=mds.lat if 'lat' in mds.coords else ref_sl.lat,
        lon=mds.lon if 'lon' in mds.coords else ref_sl.lon,
        method='nearest',
    )
    for m in PAPER_METRICS:
        if m not in mds:
            continue
        vals = mds[m].values
        cmask2 = conf_on_grid2.values
        cv = vals[cmask2 >= 0.5]
        sv = vals[cmask2 < 0.5]
        cv = cv[np.isfinite(cv)]
        sv = sv[np.isfinite(sv)]
        if len(cv) > 0:
            baseline_covered[m].append(np.median(cv))
        if len(sv) > 0:
            baseline_sparse[m].append(np.median(sv))
    if idx % 6 == 0:
        print(f"  Baseline stratified: {idx}/36")

cpc_ds2.close()
imergl_ds2.close()

stratified['IMERG-L'] = {
    'covered': {
        m: np.median(baseline_covered[m]) if baseline_covered[m] else np.nan
        for m in PAPER_METRICS
    },
    'sparse': {
        m: np.median(baseline_sparse[m]) if baseline_sparse[m] else np.nan
        for m in PAPER_METRICS
    },
}

# Print stratified results
print("\n" + "=" * 80)
print("STRATIFIED ANALYSIS: Station-covered (C >= 0.5) vs Station-sparse (C < 0.5)")
print("=" * 80)
for zone in ['covered', 'sparse']:
    zone_label = 'Station-covered' if zone == 'covered' else 'Station-sparse'
    print(f"\n--- {zone_label} ---")
    for m, ml in zip(PAPER_METRICS, PAPER_METRIC_LABELS):
        vals = [stratified[k][zone][m] for k in ['IMERG-L'] + METHOD_LABELS]
        print(f"  {ml:>7s}: IMERG-L={vals[0]:.4f} -> LS={vals[1]:.4f} -> "
              f"LSEQM={vals[2]:.4f} -> LSEQM+DL={vals[3]:.4f}")

print(f"\nNarrative: Station-covered ~{station_covered_pct:.0f}% of land pixels")

### Step 4: Table 5 -- Precipitation Percentiles

**Chapter 4.3** -- Domain-median percentiles for CPC-UNI reference and each correction stage.

In [ ]:
# ==============================================================
# Step 4: Table 5 -- Precipitation percentiles
# ==============================================================

pctl_vars_ref = ['p50_ref', 'p75_ref', 'p90_ref', 'p95_ref', 'p99_ref']
pctl_vars_test = ['p50_test', 'p75_test', 'p90_test', 'p95_test', 'p99_test']
pctl_labels = ['Q50', 'Q75', 'Q90', 'Q95', 'Q99']

# CPC reference percentiles (same across methods, use LS as source)
ref_summaries = load_and_summarize_all_dekads('ls', 'cpc', pctl_vars_ref)
ref_pctl = aggregate_summaries(ref_summaries, pctl_vars_ref)

# Test percentiles for each method
test_pctls = {}
for method, label in zip(METHODS, METHOD_LABELS):
    summaries = load_and_summarize_all_dekads(method, 'cpc', pctl_vars_test)
    test_pctls[label] = aggregate_summaries(summaries, pctl_vars_test)

print("\n" + "=" * 85)
print("TABLE 5: Domain-median precipitation percentiles (mm/day)")
print("=" * 85)
print(f"{'Pctl':>6s} | {'CPC-UNI':>10s} | {'LS':>18s} | {'LSEQM':>18s} | {'LSEQM+DL':>18s}")
print("-" * 85)

for pr, pt, pl in zip(pctl_vars_ref, pctl_vars_test, pctl_labels):
    ref_val = ref_pctl[pr]['median']
    row = f"{pl:>6s} | {ref_val:10.2f}"
    for label in METHOD_LABELS:
        test_val = test_pctls[label][pt]['median']
        if ref_val != 0:
            rel_err = (test_val - ref_val) / ref_val * 100
        else:
            rel_err = 0.0
        row += f" | {test_val:7.2f} ({rel_err:+.1f}%)"
    print(row)

### Step 5: Table 6 -- Full Reference-Test Evaluation Matrix

**Chapter 4.4** -- All reference-test combinations including uncorrected baseline.

In [ ]:
# ==============================================================
# Step 5: Table 6 -- Full 9-combo evaluation matrix
# ==============================================================

table6 = {}

# Corrected combos (9 combinations)
for ref in REFS:
    for method, label in zip(METHODS, METHOD_LABELS):
        key = f"{ref.upper()} vs {label}"
        summaries = load_and_summarize_all_dekads(method, ref, PAPER_METRICS)
        if summaries:
            table6[key] = aggregate_summaries(summaries, PAPER_METRICS)
            print(f"  {key}: {len(summaries)} dekads")
        else:
            print(f"  {key}: NO DATA")

# Add CPC baseline
table6['CPC vs IMERG-L'] = baseline_agg

print("\n" + "=" * 115)
print("TABLE 6: Domain-median metrics for all reference-test combinations")
print("=" * 115)
header = f"{'Reference':>10s} | {'Test':>12s}"
for ml in PAPER_METRIC_LABELS:
    header += f" | {ml:>7s}"
print(header)
print("-" * len(header))

for ref_label in ['CPC', 'IMERGL', 'IMERGF']:
    # Baseline row
    if ref_label == 'CPC':
        bkey = 'CPC vs IMERG-L'
        if bkey in table6:
            row = f"{'CPC-UNI':>10s} | {'IMERG-L':>12s}"
            for m in PAPER_METRICS:
                row += f" | {table6[bkey][m]['median']:7.4f}"
            print(row)

    for method, mlabel in zip(METHODS, METHOD_LABELS):
        key = f"{ref_label} vs {mlabel}"
        if key in table6:
            ref_disp = '' if ref_label == 'CPC' and mlabel != 'LS' else (
                f"{ref_label:>10s}" if mlabel == 'LS' else '')
            row = f"{ref_disp:>10s} | {mlabel:>12s}"
            for m in PAPER_METRICS:
                row += f" | {table6[key][m]['median']:7.4f}"
            print(row)
    print()  # separator between ref groups

### Step 6: Figure 4 -- CQI Spatial Distribution

**Chapter 4.2** -- Mean CQI across all 36 dekads for LS, LSEQM, LSEQM+DL.

In [ ]:
# ==============================================================
# Step 6: Figure 4 -- CQI spatial distribution maps
# ==============================================================

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print("Cartopy not available; using plain pcolormesh")

mean_cqi = {}
cqi_lat = None
cqi_lon = None

for method, label in zip(METHODS, METHOD_LABELS):
    cqi_stack = []
    for month, dekad in ALL_PERIODS:
        fpath = qualitysd_path(method, 'cpc', month, dekad)
        if not os.path.isfile(fpath):
            continue
        ds = xr.open_dataset(fpath, engine=config.NETCDF_ENGINE)
        if 'continuous_quality' in ds:
            cqi_stack.append(ds['continuous_quality'].values)
            if cqi_lat is None:
                cqi_lat = ds.lat.values
                cqi_lon = ds.lon.values
        ds.close()
    if cqi_stack:
        mean_cqi[label] = np.nanmean(cqi_stack, axis=0)
    print(f"  {label}: {len(cqi_stack)} dekads averaged")

# Plot 3-panel CQI map
subplot_kw = {'projection': ccrs.PlateCarree()} if HAS_CARTOPY else {}
fig, axes = plt.subplots(1, 3, figsize=(18, 5), subplot_kw=subplot_kw)

im = None
for i, (ax, label) in enumerate(zip(axes, METHOD_LABELS)):
    if label not in mean_cqi:
        ax.set_title(f'{label} -- no data')
        continue
    data = mean_cqi[label]
    proj_kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}
    im = ax.pcolormesh(
        cqi_lon, cqi_lat, data,
        vmin=0, vmax=1, cmap='viridis', **proj_kw,
    )
    if HAS_CARTOPY:
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.set_extent([95, 141, -11, 6])
    ax.set_title(f'({chr(97 + i)}) {label}', fontsize=12)

if im is not None:
    fig.subplots_adjust(right=0.92)
    cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
    fig.colorbar(im, cax=cbar_ax, label='CQI')

fig.suptitle('Figure 4: Mean CQI across all 36 dekadal periods', fontsize=13, y=1.02)
save_fig(fig, 'fig04_cqi_spatial.png')
plt.show()

### Step 7: Figure 5 -- CQI Improvement and Categorical Classification

**Chapter 4.2** -- (a) CQI difference LSEQM+DL minus LS, (b) categorical quality map.

In [ ]:
# ==============================================================
# Step 7: Figure 5 -- CQI improvement + categorical classification
# ==============================================================

cqi_diff = mean_cqi['LSEQM+DL'] - mean_cqi['LS']

# Statistics
valid = np.isfinite(cqi_diff)
pct_improved = np.sum(cqi_diff[valid] > 0) / np.sum(valid) * 100
pct_degraded = np.sum(cqi_diff[valid] < 0) / np.sum(valid) * 100

# Categorical classification for LSEQM+DL
cqi_dl = mean_cqi['LSEQM+DL']
cat = np.full_like(cqi_dl, np.nan)
cat[cqi_dl >= 0.8] = 4  # Excellent
cat[(cqi_dl >= 0.6) & (cqi_dl < 0.8)] = 3  # Good
cat[(cqi_dl >= 0.4) & (cqi_dl < 0.6)] = 2  # Fair
cat[cqi_dl < 0.4] = 1  # Poor

valid_cat = np.isfinite(cat)
pct_excellent = np.sum(cat[valid_cat] == 4) / np.sum(valid_cat) * 100
pct_good = np.sum(cat[valid_cat] >= 3) / np.sum(valid_cat) * 100

# Same for LS
cqi_ls = mean_cqi['LS']
cat_ls = np.full_like(cqi_ls, np.nan)
cat_ls[cqi_ls >= 0.8] = 4
cat_ls[(cqi_ls >= 0.6) & (cqi_ls < 0.8)] = 3
cat_ls[(cqi_ls >= 0.4) & (cqi_ls < 0.6)] = 2
cat_ls[cqi_ls < 0.4] = 1
valid_ls = np.isfinite(cat_ls)
pct_good_ls = np.sum(cat_ls[valid_ls] >= 3) / np.sum(valid_ls) * 100

print(f"Pixels improved (CQI diff > 0): {pct_improved:.1f}%")
print(f"Pixels degraded (CQI diff < 0): {pct_degraded:.1f}%")
print(f"LSEQM+DL: Excellent: {pct_excellent:.1f}%, Good or Excellent: {pct_good:.1f}%")
print(f"LS: Good or Excellent: {pct_good_ls:.1f}%")

# Plot
subplot_kw = {'projection': ccrs.PlateCarree()} if HAS_CARTOPY else {}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), subplot_kw=subplot_kw)

proj_kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}

# (a) Improvement map
im1 = ax1.pcolormesh(
    cqi_lon, cqi_lat, cqi_diff,
    vmin=-0.3, vmax=0.3, cmap='RdBu', **proj_kw,
)
if HAS_CARTOPY:
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax1.set_extent([95, 141, -11, 6])
ax1.set_title('(a) CQI Improvement (LSEQM+DL - LS)')
plt.colorbar(im1, ax=ax1, shrink=0.7, label='delta-CQI')

# (b) Categorical map
cat_cmap = mcolors.ListedColormap(['#d73027', '#fc8d59', '#91cf60', '#1a9850'])
bounds = [0.5, 1.5, 2.5, 3.5, 4.5]
norm = mcolors.BoundaryNorm(bounds, cat_cmap.N)
im2 = ax2.pcolormesh(
    cqi_lon, cqi_lat, cat,
    cmap=cat_cmap, norm=norm, **proj_kw,
)
if HAS_CARTOPY:
    ax2.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax2.set_extent([95, 141, -11, 6])
ax2.set_title('(b) LSEQM+DL Quality Classification')
cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.7, ticks=[1, 2, 3, 4])
cbar2.ax.set_yticklabels(['Poor', 'Fair', 'Good', 'Excellent'])

fig.suptitle(
    'Figure 5: CQI improvement and categorical classification',
    fontsize=13, y=1.02,
)
save_fig(fig, 'fig05_cqi_improvement.png')
plt.show()

### Step 8: Figure 6 -- CQI Component Score Distributions

**Chapter 4.3** -- Box plots of BSQS, DQS, TQS, CQI across all land pixels for 3 methods.

In [ ]:
# ==============================================================
# Step 8: Figure 6 -- CQI component score box plots
# ==============================================================

qa_vars = [
    'basic_statistical_quality',
    'distribution_quality',
    'temporal_quality',
    'continuous_quality',
]
qa_labels = ['(a) BSQS', '(b) DQS', '(c) TQS', '(d) CQI']

qa_data = {label: {v: [] for v in qa_vars} for label in METHOD_LABELS}

for method, label in zip(METHODS, METHOD_LABELS):
    for month, dekad in ALL_PERIODS:
        fpath = qualitysd_path(method, 'cpc', month, dekad)
        if not os.path.isfile(fpath):
            continue
        ds = xr.open_dataset(fpath, engine=config.NETCDF_ENGINE)
        for v in qa_vars:
            if v in ds:
                vals = ds[v].values.ravel()
                vals = vals[np.isfinite(vals)]
                qa_data[label][v].extend(vals.tolist())
        ds.close()

# Convert to arrays
for label in METHOD_LABELS:
    for v in qa_vars:
        qa_data[label][v] = np.array(qa_data[label][v])

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)

for i, (v, ql) in enumerate(zip(qa_vars, qa_labels)):
    ax = axes[i]
    bp_data = [qa_data[label][v] for label in METHOD_LABELS]
    bp = ax.boxplot(
        bp_data, labels=METHOD_LABELS, whis=[5, 95], showfliers=False,
        patch_artist=True,
        medianprops={'color': 'black', 'linewidth': 1.5},
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(ql, fontsize=11)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Figure 6: CQI component score distributions', fontsize=13, y=1.02)
plt.tight_layout()
save_fig(fig, 'fig06_component_boxplots.png')
plt.show()

# KS p-value statistics
print("\n--- KS p-value statistics ---")
for method, label in zip(METHODS, METHOD_LABELS):
    ks_vals = []
    for month, dekad in ALL_PERIODS:
        fpath = metricssd_path(method, 'cpc', month, dekad)
        if not os.path.isfile(fpath):
            continue
        ds = xr.open_dataset(fpath, engine=config.NETCDF_ENGINE)
        if 'ks_pvalue' in ds:
            v = ds['ks_pvalue'].values.ravel()
            v = v[np.isfinite(v)]
            ks_vals.extend(v.tolist())
        ds.close()
    ks_arr = np.array(ks_vals)
    if len(ks_arr) > 0:
        med = np.median(ks_arr)
        pct_above_05 = np.sum(ks_arr > 0.05) / len(ks_arr) * 100
        print(f"  {label}: median KS p = {med:.4f}, "
              f"{pct_above_05:.1f}% of pixels p > 0.05")

### Step 9: Table 7 + Figure 7 -- Station-Level Validation

**Chapter 4.5** -- Summary of station-level metrics + station location and NSE maps.

In [ ]:
# ==============================================================
# Step 9a: Table 7 -- Station validation summary
# ==============================================================

station_metrics = {}
for method, label in zip(METHODS, METHOD_LABELS):
    all_station_data = []
    for month, dekad in ALL_PERIODS:
        fpath = station_val_path(method, month, dekad)
        if not os.path.isfile(fpath):
            continue
        df = pd.read_csv(fpath)
        df['_month'] = month
        df['_dekad'] = dekad
        all_station_data.append(df)
    if all_station_data:
        station_metrics[label] = pd.concat(all_station_data, ignore_index=True)
        print(f"  {label}: {len(all_station_data)} periods, "
              f"{len(station_metrics[label])} rows")

# Aggregate: median across all stations x dekads for each metric
sval_metric_map = {
    'relative_bias': 'RB',
    'pearson_correlation': 'Corr',
    'rmse': 'RMSE',
    'mae': 'MAE',
    'nse': 'NSE',
    'pod': 'POD',
    'csi': 'CSI',
    'ks_pvalue': 'KS p',
}

print("\n" + "=" * 90)
print("TABLE 7: Station validation -- median metrics across BMKG stations")
print("=" * 90)
header = f"{'Method':>10s} | {'N sta':>6s}"
for ml in sval_metric_map.values():
    header += f" | {ml:>7s}"
print(header)
print("-" * len(header))

for label in METHOD_LABELS:
    if label not in station_metrics:
        continue
    df = station_metrics[label]
    # Station ID column may be 'station_id' (index) or unnamed first column
    sta_col = None
    for candidate in ['station_id', 'Station', 'ID_WMO']:
        if candidate in df.columns:
            sta_col = candidate
            break
    if sta_col is not None:
        n_stations = df[sta_col].nunique()
    else:
        n_stations = '?'
    row = f"{label:>10s} | {n_stations:>6}"
    for m in sval_metric_map:
        if m in df.columns:
            row += f" | {df[m].median():7.4f}"
        else:
            row += f" |     N/A"
    print(row)

In [ ]:
# ==============================================================
# Step 9b: Figure 7 -- Station location + per-station NSE map
# ==============================================================

from src.station_density import load_station_locations

station_df = load_station_locations(config.STATION_FILE)

subplot_kw = {'projection': ccrs.PlateCarree()} if HAS_CARTOPY else {}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), subplot_kw=subplot_kw)

proj_kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}

# (a) Station locations
if HAS_CARTOPY:
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax1.add_feature(cfeature.LAND, alpha=0.1)
    ax1.set_extent([95, 141, -11, 6])
ax1.scatter(
    station_df['Lon'], station_df['Lat'],
    s=20, c='red', alpha=0.7, zorder=5, **proj_kw,
)
ax1.set_title(f'(a) BMKG Stations (n={len(station_df)})')

# (b) Per-station NSE for LSEQM+DL (median across dekads)
if 'LSEQM+DL' in station_metrics:
    df_dl = station_metrics['LSEQM+DL']
    # Identify the station ID column
    sta_col = None
    for candidate in ['station_id', 'Station', 'ID_WMO']:
        if candidate in df_dl.columns:
            sta_col = candidate
            break
    if sta_col is not None and 'nse' in df_dl.columns:
        nse_per_station = df_dl.groupby(sta_col)['nse'].median()
        # Merge with station locations
        station_df_copy = station_df.copy()
        station_df_copy['ID_WMO'] = station_df_copy['ID_WMO'].astype(int)
        nse_per_station.index = nse_per_station.index.astype(int)
        merged = station_df_copy.set_index('ID_WMO').join(
            nse_per_station.rename('nse_med'), how='inner'
        )

        if HAS_CARTOPY:
            ax2.add_feature(cfeature.COASTLINE, linewidth=0.5)
            ax2.set_extent([95, 141, -11, 6])
        sc2 = ax2.scatter(
            merged['Lon'], merged['Lat'],
            c=merged['nse_med'], cmap='RdYlGn',
            vmin=-0.5, vmax=1.0, s=30, zorder=5, **proj_kw,
        )
        plt.colorbar(sc2, ax=ax2, shrink=0.7, label='NSE')

ax2.set_title('(b) Per-Station NSE (LSEQM+DL)')

fig.suptitle('Figure 7: Station validation overview', fontsize=13, y=1.02)
save_fig(fig, 'fig07_station_validation.png')
plt.show()

### Step 10: Figure 8 -- WMO Multi-Threshold Verification

**Chapter 4.5** -- POD, CSI, ETS vs precipitation threshold.

In [ ]:
# ==============================================================
# Step 10: Figure 8 -- WMO multi-threshold verification curves
# ==============================================================

WMO_THRESHOLDS = [1, 5, 10, 20, 50, 100, 150]
mt_metrics_list = ['pod', 'csi', 'ets']
mt_labels_list = ['(a) POD', '(b) CSI', '(c) ETS']

mt_data = {}
for method, label in zip(METHODS, METHOD_LABELS):
    all_mt = []
    for month, dekad in ALL_PERIODS:
        fpath = multi_thresh_path(method, month, dekad)
        if not os.path.isfile(fpath):
            continue
        df = pd.read_csv(fpath, index_col=0)
        all_mt.append(df)
    if all_mt:
        mt_data[label] = pd.concat(all_mt, ignore_index=True)
        print(f"  {label}: {len(all_mt)} periods")

colors = {'LS': '#1f77b4', 'LSEQM': '#ff7f0e', 'LSEQM+DL': '#2ca02c'}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, (metric, mtl) in enumerate(zip(mt_metrics_list, mt_labels_list)):
    ax = axes[i]
    for label in METHOD_LABELS:
        if label not in mt_data:
            continue
        df = mt_data[label]
        medians = []
        q25s = []
        q75s = []
        for t in WMO_THRESHOLDS:
            # Try column naming pattern: {metric}_median for summary CSVs
            med_col = f"{metric}_median"
            p25_col = f"{metric}_p25"
            p75_col = f"{metric}_p75"

            # The summary CSV has threshold_mm as index
            # When concat'd, threshold_mm becomes a regular column
            if 'threshold_mm' in df.columns:
                row = df[df['threshold_mm'] == t]
            elif df.index.name == 'threshold_mm' or t in df.index:
                row = df.loc[[t]] if t in df.index else pd.DataFrame()
            else:
                row = pd.DataFrame()

            if not row.empty and med_col in row.columns:
                medians.append(row[med_col].median())
                q25s.append(row[p25_col].median() if p25_col in row.columns else np.nan)
                q75s.append(row[p75_col].median() if p75_col in row.columns else np.nan)
            else:
                medians.append(np.nan)
                q25s.append(np.nan)
                q75s.append(np.nan)

        ax.plot(
            WMO_THRESHOLDS, medians, 'o-',
            label=label, color=colors[label],
        )
        ax.fill_between(
            WMO_THRESHOLDS, q25s, q75s,
            alpha=0.15, color=colors[label],
        )

    ax.set_xscale('log')
    ax.set_xlabel('Threshold (mm/day)')
    ax.set_title(mtl, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)

fig.suptitle('Figure 8: WMO multi-threshold verification', fontsize=13, y=1.02)
plt.tight_layout()
save_fig(fig, 'fig08_multi_threshold.png')
plt.show()

# Narrative values
print("\n--- Multi-threshold narrative values ---")
for label in METHOD_LABELS:
    if label not in mt_data:
        continue
    df = mt_data[label]
    for t in [50, 100]:
        for metric in ['csi', 'pod', 'ets']:
            med_col = f"{metric}_median"
            if 'threshold_mm' in df.columns:
                row = df[df['threshold_mm'] == t]
            else:
                row = pd.DataFrame()
            if not row.empty and med_col in row.columns:
                print(f"  {label} {metric.upper()} at {t}mm: "
                      f"{row[med_col].median():.3f}")

### Step 11: Figure 9 -- Station Density Confidence Mask

**Chapter 4.6** -- (a) Confidence mask map, (b) CQI improvement vs confidence quintile.

In [ ]:
# ==============================================================
# Step 11: Figure 9 -- Confidence mask analysis
# ==============================================================

# Load confidence mask
conf_ds = xr.open_dataset(config.CONFIDENCE_MASK_FILE, engine=config.NETCDF_ENGINE)
conf_arr = conf_ds['confidence'].values

# CQI improvement: LSEQM+DL - LSEQM
cqi_improvement = mean_cqi['LSEQM+DL'] - mean_cqi['LSEQM']

# Panel (a): confidence mask as map with Cartopy
# Panel (b): boxplot on plain axes
if HAS_CARTOPY:
    fig = plt.figure(figsize=(16, 5))
    ax1 = fig.add_subplot(1, 2, 1, projection=ccrs.PlateCarree())
    ax2 = fig.add_subplot(1, 2, 2)  # plain axes for boxplot
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# (a) Confidence mask
proj_kw = {'transform': ccrs.PlateCarree()} if HAS_CARTOPY else {}
im1 = ax1.pcolormesh(
    conf_ds.lon.values, conf_ds.lat.values, conf_arr,
    vmin=0, vmax=1, cmap='YlOrRd_r', **proj_kw,
)
if HAS_CARTOPY:
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax1.set_extent([95, 141, -11, 6])
ax1.set_title('(a) Station Density Confidence Mask')
plt.colorbar(im1, ax=ax1, shrink=0.7, label='Confidence C(x,y)')

# (b) CQI improvement vs confidence quintile
# Interpolate confidence to CQI grid
conf_interp_cqi = xr.DataArray(
    conf_arr, dims=['lat', 'lon'],
    coords={'lat': conf_ds.lat.values, 'lon': conf_ds.lon.values},
)
conf_on_cqi = conf_interp_cqi.interp(
    lat=cqi_lat, lon=cqi_lon, method='nearest'
).values

# Flatten and bin
conf_flat = conf_on_cqi.ravel()
improv_flat = cqi_improvement.ravel()
valid_mask = np.isfinite(conf_flat) & np.isfinite(improv_flat)
conf_v = conf_flat[valid_mask]
improv_v = improv_flat[valid_mask]

quintile_edges = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
quintile_labels_str = ['0.0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0']
boxplot_data = []
for lo, hi in zip(quintile_edges[:-1], quintile_edges[1:]):
    if hi < 1.0:
        mask = (conf_v >= lo) & (conf_v < hi)
    else:
        mask = (conf_v >= lo) & (conf_v <= hi)
    boxplot_data.append(improv_v[mask])

bp = ax2.boxplot(
    boxplot_data, labels=quintile_labels_str, whis=[5, 95],
    showfliers=False, patch_artist=True,
    medianprops={'color': 'black', 'linewidth': 1.5},
)
for patch in bp['boxes']:
    patch.set_facecolor('#2ca02c')
    patch.set_alpha(0.5)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_xlabel('Confidence Quintile')
ax2.set_ylabel('delta-CQI (LSEQM+DL - LSEQM)')
ax2.set_title('(b) CQI Improvement vs Confidence')
ax2.grid(axis='y', alpha=0.3)

fig.suptitle(
    'Figure 9: Station density confidence mask analysis',
    fontsize=13, y=1.02,
)
save_fig(fig, 'fig09_confidence_mask.png')
plt.show()
conf_ds.close()

### Step 12: Supplementary -- Wet vs Dry Season Performance

Wet season: October--March. Dry season: April--September.

In [ ]:
# ==============================================================
# Step 12: Seasonal summary -- Wet vs Dry
# ==============================================================

seasonal = {}
for season, season_months, season_label in [
    ('wet', WET_MONTHS, 'Wet (Oct-Mar)'),
    ('dry', DRY_MONTHS, 'Dry (Apr-Sep)'),
]:
    for method, label in zip(METHODS, METHOD_LABELS):
        cqi_vals = []
        for month, dekad in ALL_PERIODS:
            if month not in season_months:
                continue
            fpath = qualitysd_path(method, 'cpc', month, dekad)
            if not os.path.isfile(fpath):
                continue
            ds = xr.open_dataset(fpath, engine=config.NETCDF_ENGINE)
            if 'continuous_quality' in ds:
                v = ds['continuous_quality'].values.ravel()
                v = v[np.isfinite(v)]
                cqi_vals.extend(v.tolist())
            ds.close()
        seasonal[(season, label)] = np.median(cqi_vals) if cqi_vals else np.nan

print("SEASONAL CQI SUMMARY")
print(f"{'Season':>15s} | {'LS':>8s} | {'LSEQM':>8s} | {'LSEQM+DL':>8s}")
print("-" * 50)
for season, slabel in [('wet', 'Wet (Oct-Mar)'), ('dry', 'Dry (Apr-Sep)')]:
    row = f"{slabel:>15s}"
    for label in METHOD_LABELS:
        row += f" | {seasonal[(season, label)]:8.4f}"
    print(row)

### Step 13: Supplementary -- Per-Year Temporal Stability

Using timeseries mode (metricsts) to assess stability over 2001--2023.

In [ ]:
# ==============================================================
# Step 13: Temporal stability -- per-year RMSE
# ==============================================================

def metricsts_path(method, ref, month, dekad):
    dd = DEKAD_MAP[dekad]
    mm = f"{month:02d}"
    fname = f"idn_cli_metricsts_{ref}_imergl_{method}_month{mm}_dekad{dd}.nc4"
    return os.path.join(get_metrics_dir(method), fname)


yearly_data = {label: {} for label in METHOD_LABELS}

for method, label in zip(METHODS, METHOD_LABELS):
    year_rmse = {}
    for month, dekad in ALL_PERIODS:
        fpath = metricsts_path(method, 'cpc', month, dekad)
        if not os.path.isfile(fpath):
            continue
        ds = xr.open_dataset(fpath, engine=config.NETCDF_ENGINE)
        if 'rmse' not in ds or 'time' not in ds.dims:
            ds.close()
            continue
        for t_idx in range(len(ds.time)):
            yr = int(ds.time.values[t_idx])
            vals = ds['rmse'].isel(time=t_idx).values.ravel()
            vals = vals[np.isfinite(vals)]
            if len(vals) > 0:
                if yr not in year_rmse:
                    year_rmse[yr] = []
                year_rmse[yr].append(np.median(vals))
        ds.close()

    for yr in year_rmse:
        yearly_data[label][yr] = np.median(year_rmse[yr])
    print(f"  {label}: {len(year_rmse)} years")

fig, ax = plt.subplots(figsize=(12, 4))
for label in METHOD_LABELS:
    if not yearly_data[label]:
        continue
    years = sorted(yearly_data[label].keys())
    vals = [yearly_data[label][y] for y in years]
    ax.plot(years, vals, 'o-', label=label, markersize=4)

ax.set_xlabel('Year')
ax.set_ylabel('Domain-median RMSE (mm/day)')
ax.set_title('Temporal stability: per-year RMSE')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, 'fig_supp_temporal_stability.png')
plt.show()